# Bayesian Decision and Uncertainty: Skin Lesion Triage

Kaggle-ready experiment for melanoma-vs-benign triage on HAM10000. The workflow excludes `bcc` and `akiec`, uses lesion-aware splitting, compares an HSV histogram + GMM baseline with EfficientNet-B0, and evaluates asymmetric Bayesian decision thresholds plus MC Dropout uncertainty.

In [ ]:
import os
import random
import warnings
from copy import deepcopy
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from PIL import Image
from sklearn.metrics import auc, brier_score_loss, confusion_matrix, roc_auc_score, roc_curve
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from torchvision.models import EfficientNet_B0_Weights

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

GPU_COUNT = torch.cuda.device_count()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle T4 x2 often bottlenecks on CPU image loading. A larger batch helps amortize
# transfer/launch overhead. DataParallel is disabled by default because it is usually
# less efficient than one-GPU training in notebooks unless the input pipeline is fast.
USE_DATA_PARALLEL = False
BATCH_SIZE = 128 if torch.cuda.is_available() else 32
NUM_WORKERS = min(8, os.cpu_count() or 8)
PREFETCH_FACTOR = 4 if NUM_WORKERS > 0 else None

# Staged training: fit the head first, then fine-tune the last EfficientNet feature blocks.
HEAD_EPOCHS = 6
FINE_TUNE_EPOCHS = 6
UNFREEZE_LAST_BLOCKS = 2
HEAD_LR = 1e-3
FINE_TUNE_LR = 1e-5
WEIGHT_DECAY = 1e-4
MC_PASSES = 30

print(f"Using device: {device}")
print(f"CUDA devices: {GPU_COUNT}")
for i in range(GPU_COUNT):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
print(f"Batch size: {BATCH_SIZE}; workers: {NUM_WORKERS}; prefetch: {PREFETCH_FACTOR}")
print(f"Head epochs: {HEAD_EPOCHS}; fine-tune epochs: {FINE_TUNE_EPOCHS}; last blocks: {UNFREEZE_LAST_BLOCKS}")
print(f"Outputs directory: {OUTPUT_DIR.resolve()}")

## Load and Filter HAM10000

Attach Kaggle dataset `kmader/skin-cancer-mnist-ham10000` before running.

In [ ]:
BASE_PATH = "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"
CSV_PATH = os.path.join(BASE_PATH, "HAM10000_metadata.csv")
IMG_DIRS = [
    os.path.join(BASE_PATH, "HAM10000_images_part_1"),
    os.path.join(BASE_PATH, "HAM10000_images_part_2"),
]

df = pd.read_csv(CSV_PATH)
df = df[~df["dx"].isin(["bcc", "akiec"])].reset_index(drop=True)
df["target"] = (df["dx"] == "mel").astype(int)

def get_img_path(image_id):
    for img_dir in IMG_DIRS:
        path = os.path.join(img_dir, f"{image_id}.jpg")
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"Image not found: {image_id}")

df["image_path"] = df["image_id"].map(get_img_path)
print(df["dx"].value_counts())
print(f"Rows: {len(df):,}, melanoma prevalence: {df['target'].mean():.3f}")

In [ ]:
gkf = GroupKFold(n_splits=5)
train_idx, val_idx = next(gkf.split(df, df["target"], groups=df["lesion_id"]))
train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

train_groups = set(train_df["lesion_id"])
val_groups = set(val_df["lesion_id"])
assert train_groups.isdisjoint(val_groups), "Leakage: lesion_id appears in both train and validation"

# Split a calibration set from the training partition, grouped by lesion_id. The final
# validation set remains untouched for paper metrics.
gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
fit_idx, cal_idx = next(gss.split(train_df, train_df["target"], groups=train_df["lesion_id"]))
fit_df = train_df.iloc[fit_idx].reset_index(drop=True)
cal_df = train_df.iloc[cal_idx].reset_index(drop=True)
assert set(fit_df["lesion_id"]).isdisjoint(set(cal_df["lesion_id"])), "Leakage: lesion_id appears in both fit and calibration"

print(f"Fit rows: {len(fit_df):,}; calibration rows: {len(cal_df):,}; validation rows: {len(val_df):,}")
print(f"Fit prevalence: {fit_df['target'].mean():.3f}; calibration prevalence: {cal_df['target'].mean():.3f}; validation prevalence: {val_df['target'].mean():.3f}")

## Classical Baseline: HSV Histograms + GMM

In [ ]:
def extract_hsv_hist(path, bins=32):
    img = cv2.imread(path)
    if img is None:
        raise FileNotFoundError(path)
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    hist_h = cv2.calcHist([hsv], [0], None, [bins], [0, 180]).flatten()
    hist_s = cv2.calcHist([hsv], [1], None, [bins], [0, 256]).flatten()
    hist_v = cv2.calcHist([hsv], [2], None, [bins], [0, 256]).flatten()
    features = np.concatenate([hist_h, hist_s, hist_v]).astype(np.float32)
    return features / (features.sum() + 1e-7)

train_df_sub = train_df.sample(min(2000, len(train_df)), random_state=SEED)
val_df_sub = val_df.sample(min(500, len(val_df)), random_state=SEED)

X_train_gmm = np.stack([extract_hsv_hist(p) for p in train_df_sub["image_path"]])
y_train_gmm = train_df_sub["target"].to_numpy()
X_val_gmm = np.stack([extract_hsv_hist(p) for p in val_df_sub["image_path"]])
y_val_gmm = val_df_sub["target"].to_numpy()

gmm_benign = GaussianMixture(n_components=5, covariance_type="diag", random_state=SEED)
gmm_mel = GaussianMixture(n_components=5, covariance_type="diag", random_state=SEED)
gmm_benign.fit(X_train_gmm[y_train_gmm == 0])
gmm_mel.fit(X_train_gmm[y_train_gmm == 1])

prior_mel = y_train_gmm.mean()
log_benign = gmm_benign.score_samples(X_val_gmm) + np.log(1 - prior_mel + 1e-10)
log_mel = gmm_mel.score_samples(X_val_gmm) + np.log(prior_mel + 1e-10)
gmm_probs = 1 / (1 + np.exp(log_benign - log_mel))

gmm_fpr, gmm_tpr, _ = roc_curve(y_val_gmm, gmm_probs)
print(f"GMM validation AUC on subsample: {auc(gmm_fpr, gmm_tpr):.3f}")

## EfficientNet-B0 with MC Dropout

In [ ]:
class HAMDataset(Dataset):
    def __init__(self, frame, transform=None):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")
        label = torch.tensor(row["target"], dtype=torch.float32)
        if self.transform is not None:
            image = self.transform(image)
        return image, label

weights = EfficientNet_B0_Weights.IMAGENET1K_V1
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

# Keep augmentation useful but moderate. RandomRotation and ColorJitter are CPU-heavy.
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.10, contrast=0.10, saturation=0.10, hue=0.02),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

loader_kwargs = {
    "num_workers": NUM_WORKERS,
    "pin_memory": torch.cuda.is_available(),
    "persistent_workers": NUM_WORKERS > 0,
}
if NUM_WORKERS > 0:
    loader_kwargs["prefetch_factor"] = PREFETCH_FACTOR

fit_loader = DataLoader(
    HAMDataset(fit_df, transform_train),
    batch_size=BATCH_SIZE,
    shuffle=True,
    **loader_kwargs,
)
cal_loader = DataLoader(
    HAMDataset(cal_df, transform_val),
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs,
)
val_loader = DataLoader(
    HAMDataset(val_df, transform_val),
    batch_size=BATCH_SIZE,
    shuffle=False,
    **loader_kwargs,
)

In [ ]:
def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def set_backbone_trainable(model, trainable=False, last_blocks=0):
    base = unwrap_model(model)
    for param in base.features.parameters():
        param.requires_grad = False
    if trainable and last_blocks > 0:
        blocks = list(base.features.children())
        for block in blocks[-last_blocks:]:
            for param in block.parameters():
                param.requires_grad = True
    for param in base.classifier.parameters():
        param.requires_grad = True

def make_optimizer(model, lr):
    params = [p for p in model.parameters() if p.requires_grad]
    return torch.optim.AdamW(params, lr=lr, weight_decay=WEIGHT_DECAY)

def predict_probs(model, loader):
    model.eval()
    probs = []
    targets = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(images).squeeze(1)
            probs.append(torch.sigmoid(logits).float().cpu().numpy())
            targets.extend(labels.numpy())
    return np.concatenate(probs), np.asarray(targets)

def train_one_epoch(model, loader, criterion, optimizer, scaler):
    model.train()
    total_loss = 0.0
    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
            logits = model(images).squeeze(1)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(loader)

model = models.efficientnet_b0(weights=weights)
model.classifier = nn.Sequential(
    nn.Dropout(p=0.5, inplace=False),
    nn.Linear(1280, 1),
)
model = model.to(device)

if USE_DATA_PARALLEL and GPU_COUNT >= 2:
    model = nn.DataParallel(model)
    print(f"DataParallel enabled across {GPU_COUNT} GPUs")
elif GPU_COUNT >= 2:
    print("Multiple GPUs detected, but DataParallel is disabled. Using one GPU to avoid notebook input-pipeline overhead.")

# Use unweighted BCE so logits can be calibrated as posterior melanoma probabilities.
# Class imbalance is handled at decision time with asymmetric thresholds, not baked into training.
criterion = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
history = []
best_auc = -np.inf
best_state = None

set_backbone_trainable(model, trainable=False)
optimizer = make_optimizer(model, HEAD_LR)
print(f"Head training: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable params")
for epoch in range(1, HEAD_EPOCHS + 1):
    loss = train_one_epoch(model, fit_loader, criterion, optimizer, scaler)
    val_probs_epoch, val_targets_epoch = predict_probs(model, val_loader)
    val_auc_epoch = roc_auc_score(val_targets_epoch, val_probs_epoch)
    history.append({"stage": "head", "epoch": epoch, "loss": loss, "val_auc": val_auc_epoch})
    print(f"head {epoch:02d}/{HEAD_EPOCHS} - loss: {loss:.4f} - val_auc: {val_auc_epoch:.4f}")
    if val_auc_epoch > best_auc:
        best_auc = val_auc_epoch
        best_state = deepcopy(unwrap_model(model).state_dict())

set_backbone_trainable(model, trainable=True, last_blocks=UNFREEZE_LAST_BLOCKS)
optimizer = make_optimizer(model, FINE_TUNE_LR)
print(f"Fine-tuning: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} trainable params")
for epoch in range(1, FINE_TUNE_EPOCHS + 1):
    loss = train_one_epoch(model, fit_loader, criterion, optimizer, scaler)
    val_probs_epoch, val_targets_epoch = predict_probs(model, val_loader)
    val_auc_epoch = roc_auc_score(val_targets_epoch, val_probs_epoch)
    history.append({"stage": "fine_tune", "epoch": epoch, "loss": loss, "val_auc": val_auc_epoch})
    print(f"fine_tune {epoch:02d}/{FINE_TUNE_EPOCHS} - loss: {loss:.4f} - val_auc: {val_auc_epoch:.4f}")
    if val_auc_epoch > best_auc:
        best_auc = val_auc_epoch
        best_state = deepcopy(unwrap_model(model).state_dict())

unwrap_model(model).load_state_dict(best_state)
history_df = pd.DataFrame(history)
history_df.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
print(f"Best validation AUC during training: {best_auc:.4f}")
print(f"Saved {OUTPUT_DIR / 'training_history.csv'}")

## Calibration, Bayesian Thresholds, and Uncertainty

In [ ]:
def collect_logits(model, loader):
    model.eval()
    logits_all = []
    targets_all = []
    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                logits = model(images).squeeze(1)
            logits_all.append(logits.float().cpu())
            targets_all.append(labels.float())
    return torch.cat(logits_all), torch.cat(targets_all)

class PlattScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.a = nn.Parameter(torch.ones(1))
        self.b = nn.Parameter(torch.zeros(1))

    def forward(self, logits):
        return self.a * logits + self.b

def fit_platt_scaler(logits, labels):
    scaler_model = PlattScaler()
    criterion_platt = nn.BCEWithLogitsLoss()
    optimizer_platt = torch.optim.LBFGS(scaler_model.parameters(), lr=0.05, max_iter=100)

    def closure():
        optimizer_platt.zero_grad()
        loss = criterion_platt(scaler_model(logits), labels)
        loss.backward()
        return loss

    optimizer_platt.step(closure)
    return float(scaler_model.a.detach().item()), float(scaler_model.b.detach().item())

def apply_platt(logits, a, b):
    return a * logits + b

def enable_dropout(module):
    for child in module.modules():
        if child.__class__.__name__.startswith("Dropout"):
            child.train()

def mc_dropout_predict(model, loader, passes=MC_PASSES, platt_a=1.0, platt_b=0.0):
    model.eval()
    enable_dropout(model)
    all_passes = []
    targets = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device, non_blocking=True)
            targets.extend(labels.numpy())
            batch_passes = []
            for _ in range(passes):
                with torch.amp.autocast("cuda", enabled=torch.cuda.is_available()):
                    logits = model(images).squeeze(1)
                calibrated_logits = apply_platt(logits.float(), platt_a, platt_b)
                probs = torch.sigmoid(calibrated_logits).cpu().numpy()
                batch_passes.append(probs)
            all_passes.append(np.stack(batch_passes, axis=0))

    return np.concatenate(all_passes, axis=1), np.asarray(targets)

def uncertainty_from_mc(all_probs):
    predictive_mean = all_probs.mean(axis=0)
    predictive_entropy = -(
        predictive_mean * np.log(predictive_mean + 1e-10)
        + (1 - predictive_mean) * np.log(1 - predictive_mean + 1e-10)
    )
    expected_entropy = np.mean(
        -(all_probs * np.log(all_probs + 1e-10) + (1 - all_probs) * np.log(1 - all_probs + 1e-10)),
        axis=0,
    )
    mutual_information = predictive_entropy - expected_entropy
    return predictive_mean, predictive_entropy, mutual_information

cal_logits, cal_targets_tensor = collect_logits(model, cal_loader)
platt_a, platt_b = fit_platt_scaler(cal_logits, cal_targets_tensor)
print(f"Platt scaling parameters: a={platt_a:.3f}, b={platt_b:.3f}")

cal_all_probs, cal_targets = mc_dropout_predict(model, cal_loader, passes=MC_PASSES, platt_a=platt_a, platt_b=platt_b)
cal_predictive_mean, cal_predictive_entropy, cal_mutual_information = uncertainty_from_mc(cal_all_probs)

all_probs, targets = mc_dropout_predict(model, val_loader, passes=MC_PASSES, platt_a=platt_a, platt_b=platt_b)
predictive_mean, predictive_entropy, mutual_information = uncertainty_from_mc(all_probs)

val_auc = roc_auc_score(targets, predictive_mean)
brier = brier_score_loss(targets, predictive_mean)
print(f"EfficientNet MC Platt-calibrated AUC: {val_auc:.4f}")
print(f"Brier score: {brier:.4f}")

In [ ]:
def expected_calibration_error(y_true, y_prob, bins=10):
    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (y_prob > lo) & (y_prob <= hi)
        if not np.any(mask):
            continue
        empirical_rate = y_true[mask].mean()
        confidence = y_prob[mask].mean()
        ece += mask.mean() * abs(empirical_rate - confidence)
    return ece

def metrics_at_threshold(probs, y_true, tau):
    preds = (probs >= tau).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, preds).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    referral_rate = preds.mean()
    return sensitivity, specificity, precision, referral_rate, int(fn), int(fp)

def evaluate_threshold(probs, y_true, l_fn, l_fp):
    tau = l_fp / (l_fp + l_fn)
    sensitivity, specificity, precision, referral_rate, fn, fp = metrics_at_threshold(probs, y_true, tau)
    return {
        "ratio": f"{l_fn}:{l_fp}",
        "threshold": tau,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "precision": precision,
        "referral_rate": referral_rate,
        "false_negatives": fn,
        "false_positives": fp,
    }

def find_threshold_for_min_sensitivity(probs, y_true, min_sensitivity=0.98):
    rows = []
    for tau in np.linspace(0.0, 1.0, 1001):
        sensitivity, specificity, precision, referral_rate, fn, fp = metrics_at_threshold(probs, y_true, tau)
        if sensitivity >= min_sensitivity:
            rows.append((tau, sensitivity, specificity, precision, referral_rate, fn, fp))
    if not rows:
        return None
    return max(rows, key=lambda row: row[2])

def selected_threshold_row(cal_probs, cal_y, val_probs, val_y, min_sensitivity):
    selected = find_threshold_for_min_sensitivity(cal_probs, cal_y, min_sensitivity)
    if selected is None:
        return {"target_sensitivity": min_sensitivity, "selected_threshold": np.nan}
    tau, cal_sens, cal_spec, cal_prec, cal_referral, cal_fn, cal_fp = selected
    val_sens, val_spec, val_prec, val_referral, val_fn, val_fp = metrics_at_threshold(val_probs, val_y, tau)
    return {
        "target_sensitivity": min_sensitivity,
        "selected_threshold": tau,
        "cal_sensitivity": cal_sens,
        "cal_specificity": cal_spec,
        "cal_precision": cal_prec,
        "cal_referral_rate": cal_referral,
        "cal_false_negatives": cal_fn,
        "val_sensitivity": val_sens,
        "val_specificity": val_spec,
        "val_precision": val_prec,
        "val_referral_rate": val_referral,
        "val_false_negatives": val_fn,
        "val_false_positives": val_fp,
    }

def triage_zone_summary(probs, mi, y_true, low_threshold, high_threshold=0.5, uncertainty_threshold=None):
    if uncertainty_threshold is None:
        uncertainty_threshold = np.quantile(mi, 0.95)
    low_risk = (probs < low_threshold) & (mi < uncertainty_threshold)
    high_risk = probs >= high_threshold
    manual_review = ~(low_risk | high_risk)
    positives = y_true == 1
    auto_clear_n = int(low_risk.sum())
    urgent_n = int(high_risk.sum())
    review_n = int(manual_review.sum())
    auto_clear_fn = int((low_risk & positives).sum())
    urgent_tp = int((high_risk & positives).sum())
    urgent_fp = int((high_risk & ~positives).sum())
    return {
        "low_threshold": low_threshold,
        "high_threshold": high_threshold,
        "uncertainty_threshold": uncertainty_threshold,
        "auto_reassured_rate": low_risk.mean(),
        "manual_review_rate": manual_review.mean(),
        "urgent_referral_rate": high_risk.mean(),
        "auto_reassured_count": auto_clear_n,
        "manual_review_count": review_n,
        "urgent_referral_count": urgent_n,
        "false_negatives_auto_reassured": auto_clear_fn,
        "sensitivity_after_auto_reassurance": 1 - (auto_clear_fn / positives.sum() if positives.sum() else 0.0),
        "urgent_referral_precision": urgent_tp / (urgent_tp + urgent_fp) if (urgent_tp + urgent_fp) else 0.0,
    }

ece = expected_calibration_error(targets, predictive_mean)
cal_ece = expected_calibration_error(cal_targets, cal_predictive_mean)
threshold_df = pd.DataFrame([
    evaluate_threshold(predictive_mean, targets, fn_cost, fp_cost)
    for fn_cost, fp_cost in [(1, 1), (5, 1), (10, 1), (20, 1)]
])
operating_point_df = pd.DataFrame([
    selected_threshold_row(cal_predictive_mean, cal_targets, predictive_mean, targets, target_sens)
    for target_sens in [0.95, 0.98, 0.99]
])

# Use the calibration-selected >=99% sensitivity threshold for auto-reassurance.
low_threshold = float(operating_point_df.loc[operating_point_df["target_sensitivity"] == 0.99, "selected_threshold"].iloc[0])
if not np.isfinite(low_threshold):
    low_threshold = 1 / 11
uncertainty_threshold = float(np.quantile(cal_mutual_information, 0.95))
triage_zone_df = pd.DataFrame([
    triage_zone_summary(predictive_mean, mutual_information, targets, low_threshold, high_threshold=0.5, uncertainty_threshold=uncertainty_threshold)
])

summary_df = pd.DataFrame([
    {"metric": "gmm_auc_subsample", "value": auc(gmm_fpr, gmm_tpr)},
    {"metric": "efficientnet_auc", "value": val_auc},
    {"metric": "brier_score", "value": brier},
    {"metric": "ece", "value": ece},
    {"metric": "calibration_ece", "value": cal_ece},
    {"metric": "platt_a", "value": platt_a},
    {"metric": "platt_b", "value": platt_b},
    {"metric": "mc_passes", "value": MC_PASSES},
])

predictions_df = val_df[["image_id", "lesion_id", "dx", "target"]].copy()
predictions_df["prob_melanoma"] = predictive_mean
predictions_df["predictive_entropy"] = predictive_entropy
predictions_df["mutual_information"] = mutual_information
low_risk_mask = (predictive_mean < low_threshold) & (mutual_information < uncertainty_threshold)
high_risk_mask = predictive_mean >= 0.5
predictions_df["triage_zone"] = np.select(
    [low_risk_mask, high_risk_mask],
    ["low_risk_auto_reassure", "high_risk_urgent_referral"],
    default="manual_review",
)

threshold_df.to_csv(OUTPUT_DIR / "threshold_metrics.csv", index=False)
operating_point_df.to_csv(OUTPUT_DIR / "sensitivity_operating_points.csv", index=False)
triage_zone_df.to_csv(OUTPUT_DIR / "triage_zone_metrics.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "summary_metrics.csv", index=False)
predictions_df.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)

print(f"Validation ECE: {ece:.4f}; calibration ECE: {cal_ece:.4f}")
print("\nTheoretical cost-ratio thresholds on validation:")
print(threshold_df.to_string(index=False, formatters={
    "threshold": "{:.3f}".format,
    "sensitivity": "{:.3f}".format,
    "specificity": "{:.3f}".format,
    "precision": "{:.3f}".format,
    "referral_rate": "{:.3f}".format,
}))
print("\nCalibration-selected sensitivity operating points evaluated on validation:")
print(operating_point_df.to_string(index=False, formatters={
    "target_sensitivity": "{:.2f}".format,
    "selected_threshold": "{:.3f}".format,
    "cal_sensitivity": "{:.3f}".format,
    "cal_specificity": "{:.3f}".format,
    "val_sensitivity": "{:.3f}".format,
    "val_specificity": "{:.3f}".format,
    "val_referral_rate": "{:.3f}".format,
}))
print("\nUncertainty-aware triage zones on validation:")
print(triage_zone_df.to_string(index=False, formatters={
    "low_threshold": "{:.3f}".format,
    "high_threshold": "{:.3f}".format,
    "uncertainty_threshold": "{:.4f}".format,
    "auto_reassured_rate": "{:.3f}".format,
    "manual_review_rate": "{:.3f}".format,
    "urgent_referral_rate": "{:.3f}".format,
    "sensitivity_after_auto_reassurance": "{:.3f}".format,
    "urgent_referral_precision": "{:.3f}".format,
}))
print(f"Saved metrics and predictions to {OUTPUT_DIR.resolve()}")

In [ ]:
fpr, tpr, _ = roc_curve(targets, predictive_mean)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, lw=2, label=f"EfficientNet-B0 MC Platt-calibrated AUC = {auc(fpr, tpr):.3f}")
plt.plot(gmm_fpr, gmm_tpr, lw=2, label=f"HSV GMM AUC = {auc(gmm_fpr, gmm_tpr):.3f}")
plt.plot([0, 1], [0, 1], color="black", lw=1, linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve: Melanoma Triage")
plt.legend(loc="lower right")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "roc_curve_triage.png", dpi=160)
plt.savefig(OUTPUT_DIR / "roc_curve_triage_calibrated.png", dpi=160)
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(predictive_mean, mutual_information, c=targets, cmap="coolwarm", s=12, alpha=0.65)
plt.axvline(1 / 11, color="black", linestyle="--", label="10:1 threshold")
plt.xlabel("Calibrated predictive mean P(melanoma)")
plt.ylabel("Mutual information")
plt.title("MC Dropout Epistemic Uncertainty")
plt.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "uncertainty_scatter.png", dpi=160)
plt.savefig(OUTPUT_DIR / "uncertainty_scatter_calibrated.png", dpi=160)
plt.show()

In [ ]:
# Save a small grid of real validation lesions to make the dataset tangible in the report.
example_specs = [
    ("low-risk benign", predictions_df.sort_values("prob_melanoma").query("target == 0").head(1)),
    ("urgent melanoma", predictions_df.sort_values("prob_melanoma", ascending=False).query("target == 1").head(1)),
    ("near 10:1 threshold", predictions_df.iloc[(predictions_df["prob_melanoma"] - (1 / 11)).abs().argsort()].head(1)),
    ("high uncertainty", predictions_df.sort_values("mutual_information", ascending=False).head(1)),
]

fig, axes = plt.subplots(1, len(example_specs), figsize=(12, 3.2))
path_lookup = val_df.set_index("image_id")["image_path"].to_dict()

for ax, (role, frame) in zip(axes, example_specs):
    row = frame.iloc[0]
    image = Image.open(path_lookup[row["image_id"]]).convert("RGB")
    ax.imshow(image)
    ax.axis("off")
    ax.set_title(
        f"{role}\n{row['image_id']} / {row['dx']}\n"
        f"p={row['prob_melanoma']:.3f}, MI={row['mutual_information']:.3f}",
        fontsize=9,
    )

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "example_lesions_grid.png", dpi=180)
plt.show()

## Outputs

The notebook writes all downloadable artifacts to `/kaggle/working/outputs/`:

- `training_history.csv`
- `summary_metrics.csv`
- `threshold_metrics.csv`
- `sensitivity_operating_points.csv`
- `triage_zone_metrics.csv`
- `validation_predictions.csv`
- `roc_curve_triage.png`
- `roc_curve_triage_calibrated.png`
- `uncertainty_scatter.png`
- `uncertainty_scatter_calibrated.png`
- `example_lesions_grid.png`

Download the `outputs/` folder from Kaggle and copy it back into `experiments/outputs/` so the paper can be updated with observed metrics.

In [ ]:
from IPython.display import FileLink, FileLinks, display
import shutil

output_dir = OUTPUT_DIR.resolve()
archive_path = shutil.make_archive(str(output_dir), "zip", root_dir=output_dir)

print(f"Output directory: {output_dir}")
print(f"Created archive: {archive_path}")